# Lag Plots and Autocorrelation

Lag plots and the autocorrelation function (ACF) are the most important diagnostic tools in time series analysis. They reveal the internal structure of a series — trend, seasonality, and noise — by measuring how each observation relates to its own past values.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from pathlib import Path

plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
DATA_DIR = Path('../data')

### Load Data

In [ ]:
# Airline passengers — monthly, with trend + seasonality
airline = pd.read_csv(
    DATA_DIR / 'airline_passengers.csv',
    index_col='Month',
    parse_dates=True
)
airline.columns = ['Passengers']
airline.index.freq = 'MS'
airline.head()

In [ ]:
# Daily total female births — relatively noisy, little structure
births = pd.read_csv(
    DATA_DIR / 'DailyTotalFemaleBirths.csv',
    index_col='Date',
    parse_dates=True
)
births.columns = ['Births']
births.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(airline.index, airline['Passengers'])
axes[0].set_title('Airline Passengers')
axes[0].set_ylabel('Thousands of Passengers')

axes[1].plot(births.index, births['Births'])
axes[1].set_title('Daily Total Female Births (1959)')
axes[1].set_ylabel('Number of Births')

plt.tight_layout()
plt.show()

---

## Lag Plots

A **lag plot** is a scatterplot of $y_t$ versus $y_{t-k}$ for a chosen lag $k$.

- If there is a strong linear relationship in the scatter, the series is autocorrelated at that lag.
- A shapeless cloud means little or no autocorrelation at that lag.

By looking at multiple lags simultaneously, we can spot both short-range dependence (trend) and long-range dependence (seasonality).

### Lag Plots for Airline Passengers (Lags 1--9)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for i, lag in enumerate(range(1, 10)):
    ax = axes[i // 3, i % 3]
    ax.scatter(
        airline['Passengers'].shift(lag),
        airline['Passengers'],
        alpha=0.5, s=20, edgecolors='k', linewidths=0.3
    )
    ax.set_xlabel(f'$y_{{t-{lag}}}$')
    ax.set_ylabel('$y_t$')
    ax.set_title(f'Lag {lag}')

fig.suptitle('Lag Plots — Airline Passengers', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Observations:**

- **Lag 1** shows the tightest positive relationship — consecutive months are strongly correlated (trend + momentum).
- As the lag increases, the scatter widens, but a positive relationship persists through lag 5 or so.
- At **lag 6**, the relationship weakens and may even tilt negative — peaks are now aligned with troughs half a year earlier.

### Extended Lag Plots — Looking for Seasonality

In [ ]:
# Show lags 6, 12, 18, 24 to highlight the seasonal signature
seasonal_lags = [6, 12, 18, 24]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, lag in zip(axes, seasonal_lags):
    ax.scatter(
        airline['Passengers'].shift(lag),
        airline['Passengers'],
        alpha=0.5, s=20, edgecolors='k', linewidths=0.3
    )
    ax.set_xlabel(f'$y_{{t-{lag}}}$')
    ax.set_ylabel('$y_t$')
    ax.set_title(f'Lag {lag}')

fig.suptitle('Seasonal Lag Plots — Airline Passengers', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Lag 12** shows a strong positive relationship — the same month one year ago is a good predictor. **Lag 24** is similar (two years ago). This is the hallmark of **12-month seasonality**. Lag 6, by contrast, aligns each month with the month half a cycle away, producing a weaker or negative relationship.

### Lag Plots for Daily Births (Low Autocorrelation)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for i, lag in enumerate(range(1, 10)):
    ax = axes[i // 3, i % 3]
    ax.scatter(
        births['Births'].shift(lag),
        births['Births'],
        alpha=0.4, s=15, edgecolors='k', linewidths=0.3
    )
    ax.set_xlabel(f'$y_{{t-{lag}}}$')
    ax.set_ylabel('$y_t$')
    ax.set_title(f'Lag {lag}')

fig.suptitle('Lag Plots — Daily Female Births', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

The births data shows a much more diffuse cloud at every lag — there is very little autocorrelation. This series behaves much closer to noise.

---

## The Autocorrelation Function (ACF)

The lag plot gives a visual impression. The **autocorrelation function** quantifies it.

The sample autocorrelation at lag $k$ is:

$$
r_k = \frac{\sum_{t=k+1}^{T}(y_t - \bar{y})(y_{t-k} - \bar{y})}{\sum_{t=1}^{T}(y_t - \bar{y})^2}
$$

Key properties:

- $r_0 = 1$ always (a series is perfectly correlated with itself at lag 0)
- $-1 \leq r_k \leq 1$
- $r_1$: correlation between consecutive observations
- $r_k$: correlation between observations $k$ periods apart

### Manual ACF Calculation

In [ ]:
def manual_acf(series, max_lag=20):
    """Compute the sample autocorrelation function manually."""
    y = series.dropna().values
    y_bar = y.mean()
    denominator = np.sum((y - y_bar) ** 2)
    acf_values = []
    for k in range(max_lag + 1):
        numerator = np.sum((y[k:] - y_bar) * (y[:len(y) - k] - y_bar))
        acf_values.append(numerator / denominator)
    return np.array(acf_values)

acf_manual = manual_acf(airline['Passengers'], max_lag=20)
print('Lag  |  ACF')
print('-----|------')
for k, val in enumerate(acf_manual[:13]):
    print(f'  {k:2d} | {val:+.4f}')

Notice how $r_1$ is high (strong consecutive correlation due to trend), and $r_{12}$ is also elevated (seasonal correlation).

---

## ACF Plots (Correlograms)

An **ACF plot** (also called a **correlogram**) displays $r_k$ as vertical bars for each lag $k$. The dashed horizontal lines mark the **significance bounds**:

$$
\pm \frac{1.96}{\sqrt{T}}
$$

where $T$ is the number of observations. If an autocorrelation falls within these bounds, it is not significantly different from zero at the 95% confidence level.

In [ ]:
T = len(airline)
sig_bound = 1.96 / np.sqrt(T)
print(f'Number of observations: {T}')
print(f'95% significance bound: +/-{sig_bound:.4f}')

### ACF Plot — Airline Passengers

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
plot_acf(airline['Passengers'], lags=40, ax=ax, title='')
ax.set_title('ACF — Airline Passengers', fontsize=13)
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation $r_k$')
plt.tight_layout()
plt.show()

---

## Interpreting ACF Patterns

The shape of the ACF tells you what kind of structure is present in the data.

### Pattern 1: Trend

When a series has a trend, the ACF values **start large and decay slowly**. They remain positive for many lags because nearby observations share the same upward (or downward) trajectory.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(airline.index, airline['Passengers'])
axes[0].set_title('Airline Passengers — Clear Upward Trend')
axes[0].set_ylabel('Thousands of Passengers')

plot_acf(airline['Passengers'], lags=40, ax=axes[1], title='')
axes[1].set_title('ACF — Slow Decay (Trend Signature)')
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('$r_k$')

plt.tight_layout()
plt.show()

The ACF stays positive for many lags and decays gradually — this is the fingerprint of a trend.

### Pattern 2: Seasonality

Seasonality produces **peaks at the seasonal lags** (12, 24, 36 for monthly data). The ACF shows a periodic wave.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
plot_acf(airline['Passengers'], lags=48, ax=ax, title='')
ax.set_title('ACF — Airline Passengers (48 lags)', fontsize=13)
ax.set_xlabel('Lag')
ax.set_ylabel('$r_k$')

# Annotate seasonal peaks
for seasonal_lag in [12, 24, 36, 48]:
    if seasonal_lag <= 48:
        ax.axvline(x=seasonal_lag, color='red', linestyle=':', alpha=0.5)
        ax.text(seasonal_lag, 0.95, f'  lag {seasonal_lag}', color='red',
                fontsize=8, transform=ax.get_xaxis_transform())

plt.tight_layout()
plt.show()

The red dashed lines mark lags 12, 24, 36, and 48. Notice how the ACF has **local peaks** at these multiples of 12 — this is the scalloped pattern characteristic of **trend + seasonality** combined.

### Pattern 3: No Pattern (Near-Noise)

When a series has little autocorrelation, almost all ACF values fall **within the significance bounds**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(births.index, births['Births'])
axes[0].set_title('Daily Female Births — Little Visible Structure')
axes[0].set_ylabel('Number of Births')

plot_acf(births['Births'], lags=40, ax=axes[1], title='')
axes[1].set_title('ACF — Mostly Within Bounds')
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('$r_k$')

plt.tight_layout()
plt.show()

Most of the ACF values hover near zero and fall within the significance bounds. This series has very weak autocorrelation — it is close to (but not quite) noise.

---

## White Noise

**White noise** is a time series where all observations are uncorrelated:

- $r_k \approx 0$ for all $k > 0$
- No trend, no seasonality, no pattern
- Each value is drawn independently

For white noise, we expect **95% of the ACF spikes** to lie within $\pm 1.96 / \sqrt{T}$.

If your data (or your model's residuals) look like white noise, **there is nothing left to forecast** — the signal has been fully captured.

In [ ]:
# Generate synthetic white noise
rng = np.random.default_rng(42)
white_noise = pd.Series(rng.standard_normal(200), name='White Noise')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(white_noise, linewidth=0.8)
axes[0].set_title('Synthetic White Noise')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Value')

plot_acf(white_noise, lags=40, ax=axes[1], title='')
axes[1].set_title('ACF — White Noise')
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('$r_k$')

plt.tight_layout()
plt.show()

All the ACF spikes (except lag 0) fall comfortably within the blue significance bounds. This is exactly what we expect from a series with no autocorrelation.

In [ ]:
# Verify: count how many lags exceed the significance bound
from statsmodels.tsa.stattools import acf

acf_vals = acf(white_noise, nlags=40)
T_wn = len(white_noise)
bound = 1.96 / np.sqrt(T_wn)

# Exclude lag 0
n_significant = np.sum(np.abs(acf_vals[1:]) > bound)
n_total = len(acf_vals[1:])

print(f'Significance bound: +/-{bound:.4f}')
print(f'Lags exceeding bound: {n_significant} / {n_total}')
print(f'Expected by chance (5%): ~{0.05 * n_total:.1f}')

Roughly 5% of lags exceed the bound purely by chance — exactly as the theory predicts.

---

## ACF After Differencing

Differencing removes structure from a series. We can use the ACF to verify that differencing has been effective:

1. **First difference** $\Delta y_t = y_t - y_{t-1}$ removes trend
2. **Seasonal difference** $\Delta_{12} y_t = y_t - y_{t-12}$ removes seasonality
3. **Both** removes trend + seasonality

This is a preview of the reasoning behind ARIMA / SARIMA models (Chapter 08).

### Original Series

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(airline.index, airline['Passengers'])
axes[0].set_title('Original — Airline Passengers')
axes[0].set_ylabel('Passengers')

plot_acf(airline['Passengers'], lags=40, ax=axes[1], title='')
axes[1].set_title('ACF — Original (trend + seasonality)')
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('$r_k$')

plt.tight_layout()
plt.show()

### After First Differencing — Trend Removed

In [ ]:
airline_diff1 = airline['Passengers'].diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(airline_diff1.index, airline_diff1)
axes[0].set_title('First Difference $\Delta y_t$')
axes[0].set_ylabel('Change in Passengers')
axes[0].axhline(0, color='black', linewidth=0.5)

plot_acf(airline_diff1, lags=40, ax=axes[1], title='')
axes[1].set_title('ACF — After diff(1): trend gone, seasonality remains')
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('$r_k$')

plt.tight_layout()
plt.show()

The slow decay is gone — the trend has been removed. But clear **peaks at lags 12, 24, 36** remain, confirming that the seasonal component is still present.

### After First + Seasonal Differencing — Approaching White Noise

In [ ]:
airline_diff1_12 = airline['Passengers'].diff(1).diff(12).dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(airline_diff1_12.index, airline_diff1_12)
axes[0].set_title('First + Seasonal Difference $\Delta \Delta_{12} y_t$')
axes[0].set_ylabel('Value')
axes[0].axhline(0, color='black', linewidth=0.5)

plot_acf(airline_diff1_12, lags=40, ax=axes[1], title='')
axes[1].set_title('ACF — After diff(1) + diff(12): near white noise')
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('$r_k$')

plt.tight_layout()
plt.show()

After both first and seasonal differencing, the ACF is much closer to the white-noise pattern. Most spikes fall within the significance bounds. The series is now approximately **stationary** — a prerequisite for ARIMA modelling.

A few significant spikes remain (e.g., at lags 1 and 12), which is why SARIMA models include MA and seasonal MA terms to capture this remaining structure.

### Side-by-Side ACF Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

plot_acf(airline['Passengers'], lags=40, ax=axes[0], title='')
axes[0].set_title('Original')
axes[0].set_xlabel('Lag')
axes[0].set_ylabel('$r_k$')

plot_acf(airline_diff1, lags=40, ax=axes[1], title='')
axes[1].set_title('After diff(1)')
axes[1].set_xlabel('Lag')

plot_acf(airline_diff1_12, lags=40, ax=axes[2], title='')
axes[2].set_title('After diff(1) + diff(12)')
axes[2].set_xlabel('Lag')

fig.suptitle('Effect of Differencing on the ACF', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

This progression — from slowly decaying ACF to near-white-noise — is exactly how we decide the differencing orders ($d$ and $D$) for ARIMA/SARIMA models.

---

## Summary

| Tool | What it does | What it tells you |
|------|-------------|-------------------|
| **Lag plot** | Scatterplot of $y_t$ vs $y_{t-k}$ | Visualises pairwise dependencies at specific lags |
| **ACF** | $r_k$ for all lags $k$ | Quantifies the strength and sign of those dependencies |
| **Significance bounds** | $\pm 1.96 / \sqrt{T}$ | Distinguishes real autocorrelation from noise |

**Reading the ACF:**

- **Slow decay** from a high initial value indicates **trend**
- **Periodic peaks** (e.g., at lags 12, 24, 36) indicate **seasonality**
- **Both** produce a scalloped, slowly decaying pattern
- **All values within bounds** (white noise) means **nothing left to model**

**Practical use:**

- The ACF pattern guides model selection (AR, MA, ARIMA, SARIMA)
- After fitting a model, check the **residuals' ACF** — it should look like white noise
- Differencing progressively removes structure; the ACF confirms when enough differencing has been applied